# 06 — Análise dos resultados

Objetivo: carregar os resultados dos experimentos, calcular estatísticas agregadas e comparar comportamento por tamanho de instância, Cmax, tempo e diversidade.

In [ ]:
import os
import sys
import json

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

caminho_raiz = os.path.abspath(os.path.join(os.getcwd(), ".."))
if caminho_raiz not in sys.path:
    sys.path.insert(0, caminho_raiz)

In [ ]:
def carregar_configuracao(caminho_config):
    with open(caminho_config, "r", encoding="utf-8") as arquivo:
        return yaml.safe_load(arquivo)


def carregar_tabela_resultados(caminho_csv):
    if not os.path.isfile(caminho_csv):
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho_csv}. "
            "Rode primeiro o notebook 05 para gerar o resumo Large."
        )

    return pd.read_csv(caminho_csv)


def carregar_historicos_resultados(caminho_json):
    if not os.path.isfile(caminho_json):
        return []

    with open(caminho_json, "r", encoding="utf-8") as arquivo:
        return json.load(arquivo)


def calcular_gap_percentual(valor, melhor_valor):
    if melhor_valor == 0:
        return 0.0
    return 100.0 * (valor - melhor_valor) / melhor_valor

In [ ]:
caminho_config = os.path.join(caminho_raiz, "config.yaml")
config = carregar_configuracao(caminho_config)

diretorio_outputs = os.path.join(caminho_raiz, config["caminhos"]["outputs"])
caminho_csv_large = os.path.join(diretorio_outputs, "resumo_large.csv")
caminho_json_large = os.path.join(diretorio_outputs, "resumo_large_com_historicos.json")

tabela_resultados = carregar_tabela_resultados(caminho_csv_large)
historicos_resultados = carregar_historicos_resultados(caminho_json_large)

tabela_resultados.head()

In [ ]:
tabela_estatisticas = (
    tabela_resultados
    .groupby(["instancia", "numero_jobs", "numero_maquinas"], as_index=False)
    .agg(
        execucoes=("cmax_best", "count"),
        cmax_medio=("cmax_best", "mean"),
        cmax_desvio_padrao=("cmax_best", "std"),
        cmax_melhor=("cmax_best", "min"),
        cmax_pior=("cmax_best", "max"),
        tempo_medio_segundos=("tempo_segundos", "mean"),
        alpha_medio=("alpha_medio", "mean"),
        beta_medio=("beta_medio", "mean"),
        diversidade_media=("diversidade_media", "mean"),
    )
)

tabela_estatisticas["gap_pior_melhor_percentual"] = tabela_estatisticas.apply(
    lambda linha: calcular_gap_percentual(linha["cmax_pior"], linha["cmax_melhor"]),
    axis=1,
)

tabela_estatisticas = tabela_estatisticas.sort_values(
    ["numero_jobs", "numero_maquinas", "cmax_medio"]
)

tabela_estatisticas

In [ ]:
melhores_por_instancia = (
    tabela_resultados
    .sort_values(["instancia", "cmax_best", "tempo_segundos"])
    .groupby("instancia", as_index=False)
    .first()
)

melhores_por_instancia[
    [
        "instancia",
        "numero_jobs",
        "numero_maquinas",
        "indice_execucao",
        "semente",
        "cmax_best",
        "tempo_segundos",
        "alpha_medio",
        "beta_medio",
        "diversidade_media",
    ]
]

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(tabela_estatisticas["instancia"], tabela_estatisticas["cmax_medio"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Cmax médio")
plt.title("Cmax médio por instância Large")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(tabela_estatisticas["instancia"], tabela_estatisticas["tempo_medio_segundos"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Tempo médio (s)")
plt.title("Tempo médio de execução por instância Large")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(
    tabela_estatisticas["diversidade_media"],
    tabela_estatisticas["cmax_medio"],
    s=80,
)

for _, linha in tabela_estatisticas.iterrows():
    plt.annotate(
        linha["instancia"],
        (linha["diversidade_media"], linha["cmax_medio"]),
        fontsize=8,
        alpha=0.8,
    )

plt.xlabel("Diversidade estrutural média")
plt.ylabel("Cmax médio")
plt.title("Relação entre diversidade estrutural e Cmax")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
if historicos_resultados:
    melhor_historico = min(historicos_resultados, key=lambda linha: linha["cmax_best"])

    plt.figure(figsize=(8, 5))
    plt.plot(melhor_historico["historico_cmax_best"], linewidth=2)
    plt.xlabel("Geração")
    plt.ylabel("Cmax_best")
    plt.title(f"Melhor curva de convergência — {melhor_historico['instancia']}")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(melhor_historico["historico_alpha"], label="alpha")
    plt.plot(melhor_historico["historico_beta"], label="beta")
    plt.xlabel("Geração")
    plt.ylabel("Valor")
    plt.title(f"Controle fuzzy — {melhor_historico['instancia']}")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print("Históricos não encontrados. A análise agregada continua disponível pela tabela CSV.")

In [ ]:
ranking_instancias = tabela_estatisticas.copy()
ranking_instancias["posicao_cmax"] = ranking_instancias["cmax_medio"].rank(method="dense")
ranking_instancias["posicao_tempo"] = ranking_instancias["tempo_medio_segundos"].rank(method="dense")
ranking_instancias["indice_composto"] = ranking_instancias["posicao_cmax"] + ranking_instancias["posicao_tempo"]

ranking_instancias = ranking_instancias.sort_values("indice_composto")

ranking_instancias[
    [
        "instancia",
        "cmax_medio",
        "cmax_melhor",
        "tempo_medio_segundos",
        "diversidade_media",
        "posicao_cmax",
        "posicao_tempo",
        "indice_composto",
    ]
]

In [ ]:
## Conclusão

A análise compara as instâncias Large por qualidade da solução, estabilidade entre execuções, tempo médio e sinais internos do PBIL-Fuzzy.
Para o relatório final, use principalmente `cmax_melhor`, `cmax_medio`, `cmax_desvio_padrao`, `tempo_medio_segundos` e as curvas de convergência da melhor execução.